## Interactive visuals

In [27]:
# saving/accessing files
from pathlib import Path

# editing files
import geopandas as gpd
import pandas as pd
import json

# maths functions
import numpy as np

# interactive map
import folium as fl
import webbrowser
import branca.colormap as cm

## Saving petitions data to variables

In [28]:
petitions_region_df = pd.read_csv("data/Top_topics_by_region.csv")
petitions_con_df = pd.read_csv("data/Top_topics_by_constituency.csv")
petitions_df = pd.read_csv("data/Top_topics.csv")

In [29]:
# # lopping through each file to calculate the weighted topics. Weighted means that the topic count is weighted in context to how 
# # many petitions that constituency has signed overall such that south east is not as disproportionately represented.
# for i in range (16):
#     # set each weighted topic count to 0 to add to later
#     petitions_region_df[f"Weighted Topic {i+1}"] = 0

# for i in range(len(petitions_region_df)):
#     # remove non_numeric columns
#     topic_interest_cols = petitions_region_df.drop(columns=['Unnamed: 0', 'Region', 'Year'])
#     # sum each numeric col in that row to get overall interest for that constituency in that year
#     total_regional_interest = topic_interest_cols.sum(axis=1, skipna=True)
#     for j in range(16):
#         # divide each topic by the overall to get each topic's score
#         petitions_region_df[f"Weighted Topic {j+1}"][i] = petitions_region_df.iloc[i][f"Topic {j+1} Count"] / total_regional_interest[i]

# for i in range (16):
#     # set each weighted topic count to 0 to add to later
#     petitions_con_df[f"Weighted Topic {i+1}"] = 0

# for i in range(len(petitions_con_df)):
#     # remove non_numeric columns
#     topic_interest_cols = petitions_con_df.drop(columns=['Unnamed: 0', 'Constituency', 'Year'])
#     # sum each numeric col in that row to get overall interest for that constituency in that year
#     total_con_interest = topic_interest_cols.sum(axis=1, skipna=True)
#     for j in range(16):
#         # divide each topic by the overall to get each topic's score
#         petitions_con_df[f"Weighted Topic {j+1}"][i] = petitions_con_df.iloc[i][f"Topic {j+1} Count"] / total_con_interest[i]

In [30]:

# petitions_con_df = petitions_con_df.drop(["Topic 1 Count", "Topic 2 Count", "Topic 3 Count", "Topic 4 Count", "Topic 5 Count", "Topic 6 Count", "Topic 7 Count", "Topic 8 Count", 
#                                           "Topic 9 Count", "Topic 10 Count", "Topic 11 Count", "Topic 12 Count", "Topic 13 Count", "Topic 14 Count", "Topic 15 Count", 
#                                           "Topic 16 Count"], axis=1)

In [31]:
# filtering petitions df by year
petitions_region_2015 = petitions_region_df[petitions_region_df["Year"]==2015]
petitions_region_2016 = petitions_region_df[petitions_region_df["Year"]==2016]
petitions_region_2017 = petitions_region_df[petitions_region_df["Year"]==2017]
petitions_region_2018 = petitions_region_df[petitions_region_df["Year"]==2018]
petitions_region_2019 = petitions_region_df[petitions_region_df["Year"]==2019]
petitions_region_2020 = petitions_region_df[petitions_region_df["Year"]==2020]
petitions_region_2021 = petitions_region_df[petitions_region_df["Year"]==2021]
petitions_region_2022 = petitions_region_df[petitions_region_df["Year"]==2022]
petitions_region_2023 = petitions_region_df[petitions_region_df["Year"]==2023]
petitions_region_2024 = petitions_region_df[petitions_region_df["Year"]==2024]
petitions_region_2025 = petitions_region_df[petitions_region_df["Year"]==2025]

# filtering petitions df by year
petitions_con_2015 = petitions_con_df[petitions_con_df["Year"]==2015]
petitions_con_2016 = petitions_con_df[petitions_con_df["Year"]==2016]
petitions_con_2017 = petitions_con_df[petitions_con_df["Year"]==2017]
petitions_con_2018 = petitions_con_df[petitions_con_df["Year"]==2018]
petitions_con_2019 = petitions_con_df[petitions_con_df["Year"]==2019]
petitions_con_2020 = petitions_con_df[petitions_con_df["Year"]==2020]
petitions_con_2021 = petitions_con_df[petitions_con_df["Year"]==2021]
petitions_con_2022 = petitions_con_df[petitions_con_df["Year"]==2022]
petitions_con_2023 = petitions_con_df[petitions_con_df["Year"]==2023]
petitions_con_2024 = petitions_con_df[petitions_con_df["Year"]==2024]
petitions_con_2025 = petitions_con_df[petitions_con_df["Year"]==2025]

## Interactive map

In [32]:
# creating functions for map layers and colours
def add_colour_col(topic_num, filtered_file):
    """ 
    function to add a column to a dataframe transferring a numeric value to a shade on a colour spectrum
    :param topic_num: integer between 1 and 16 which indicates which topic to give a colour bar for
    :param filtered_file: relevant geojson file that has been merged with petitions data
    :output: dataframe with filtered file and additional colour column
    """

    non_null_file = filtered_file[filtered_file[f"Topic {topic_num} Count"]>0]

    # min / max values - colour gradient and map bounds
    vmin = non_null_file[f"Topic {topic_num} Count"].min()
    vmax = non_null_file[f"Topic {topic_num} Count"].max()

    # assign every variable value a colour from blue-yellow
    colourmap = cm.LinearColormap(["#0D1856", "#359A61", "#FBFF00"], vmin=vmin, vmax=vmax)
    
    # Vectorized color mapping
    colours = np.where(
        pd.isna(filtered_file[f"Topic {topic_num} Count"].values),
        "#cccccc",  # Default for NaN
        [colourmap(v) if not np.isnan(v) else "#cccccc" for v in filtered_file[f"Topic {topic_num} Count"].values]
    )

    # adding the column to the df
    filtered_file[f"Topic {topic_num} Colour"] = colours

    return filtered_file


# not realistic but shows how it's done: for each variable, create a layer.
# the show argument controls which layer is open on map load.
def create_hex_layergroup_year(filtered_file, show, topic_num, topic_name, year):
    """ 
    Creates a layer to be added to the interactive html map
    :param filtered_file: relevant geojson file that has been merged with petitions data
    :param show: boolean, if true will show the layer automatically when first loaded
    :param topic_num: integer between 1 and 16 which indicates which topic the layer will be added for
    :param topic_name: string with the title of the topic for the tooltip
    :param year: integer between 2015-2025 for the year to label each layer
    """
    filtered_file = add_colour_col(topic_num, filtered_file)
    #fl.GeoJson creates a JSON Feature for each dataframe row and the param style_function takes a JSON Feature and returns HTML style properties to apply 
    this_layer = fl.GeoJson(
        data=filtered_file[["Constituency", f"Topic {topic_num} Count", f"Topic {topic_num} Colour", "geometry"]],
        name=topic_name,
        style_function=lambda feature: {
            "fillColor": feature["properties"][f"Topic {topic_num} Colour"],
            "fillOpacity": 1,
            "weight": 0.5,
            "color": "#000000",
        },
        tooltip=fl.GeoJsonTooltip(["Constituency", f"Topic {topic_num} Count"], aliases=["Constituency", topic_name]),
        highlight_function=lambda feature: {"fillColor": "#ffffff", "weight": 0.75}
    )

    # leaflet assumes layers are overlaid on top but we want them mutally exclusive.
    # use LayerGroups with overlay = False to force this
    this_layer_grouped = fl.FeatureGroup(name=year, overlay=False, show=show).add_child(this_layer)

    return this_layer_grouped

def create_hex_layergroup_topic(filtered_file, show, topic_num, topic_name):
    """ 
    Creates a layer to be added to the interactive html map
    :param filtered_file: relevant geojson file that has been merged with petitions data
    :param show: boolean, if true will show the layer automatically when first loaded
    :param topic_num: integer between 1 and 16 which indicates which topic the layer will be added for
    :param topic_name: string with the title of the topic for the tooltip and the layer label
    """
    
    filtered_file = add_colour_col(topic_num, filtered_file)

    #fl.GeoJson creates a JSON Feature for each dataframe row and the param style_function takes a JSON Feature and returns HTML style properties to apply 
    this_layer = fl.GeoJson(
        data=filtered_file[["Constituency", f"Topic {topic_num} Count", f"Topic {topic_num} Colour", "geometry"]],
        name=topic_name,
        style_function=lambda feature: {
            "fillColor": feature["properties"][f"Topic {topic_num} Colour"],
            "fillOpacity": 1,
            "weight": 0.5,
            "color": "#000000",
        },
        tooltip=fl.GeoJsonTooltip(["Constituency", f"Topic {topic_num} Count"], aliases=["Constituency", topic_name]),
        highlight_function=lambda feature: {"fillColor": "#ffffff", "weight": 0.75}
    )

    # leaflet assumes layers are overlaid on top but we want them mutally exclusive.
    # use LayerGroups with overlay = False to force this
    this_layer_grouped = fl.FeatureGroup(name=topic_name, overlay=False, show=show).add_child(this_layer)

    return this_layer_grouped

In [33]:
# functions to convert hexjson to geojson
def hexjson_to_geojson(hexjson_file, geojson_file, hex_width=0.00117, hex_height=0.00135):
    """
    Convert HexJSON format to GeoJSON format.
    :param hexjson_file: Path to input HexJSON file
    :param geojson_file: Path to output GeoJSON file
    :param hex_width: Width between hex centers (default 0.00117)
    :param hex_height: Height between hex centers (default 0.00135)
    """
    
    # Read HexJSON file
    with open(hexjson_file, 'r', encoding='utf-8') as f:
        hexjson = json.load(f)
    
    # Get the hexes data
    hexes = hexjson.get('hexes', {})
    layout = hexjson.get('layout', 'odd-r')
    
    # Initialize GeoJSON structure matching the previous format
    geojson = {
        "type": "FeatureCollection",
        "name": "constituency_results_2024_hex",
        "crs": {
            "type": "name",
            "properties": {
                "name": "urn:ogc:def:crs:OGC:1.3:CRS84"
            }
        },
        "features": []
    }
    
    # Process each constituency
    for constituency_id, hex_data in hexes.items():

        # Extract hex grid coordinates
        q = hex_data.get('q', 0)
        r = hex_data.get('r', 0)
        
        # Generate hexagon coordinates matching the example format
        hex_coords = create_hexagon_coordinates_matched(q, r, hex_width, hex_height, layout)
        
        # Create GeoJSON feature matching the example structure
        feature = {
            "type": "Feature",
            "properties": {
                "constituency_name": hex_data.get('n', ''),
                "id": constituency_id,
                "q": q,
                "r": r,
                "region": hex_data.get('region', ''),
                "colour": hex_data.get('colour', '')
            },
            "geometry": {
                "type": "Polygon",
                "coordinates": [hex_coords]
            }
        }
        
        geojson['features'].append(feature)
    
    # Write GeoJSON file
    with open(geojson_file, 'w', encoding='utf-8') as f:
        json.dump(geojson, f, indent=2, ensure_ascii=False)
    
    print(f"Converted {len(geojson['features'])} constituencies")
    print(f"Output saved to: {geojson_file}")


def create_hexagon_coordinates_matched(q, r, width, height, layout='odd-r'):
    """
    Create hexagon polygon coordinates.
    :param q: Column coordinate
    :param r: Row coordinate  
    :param width: Width between hex centers
    :param height: Height between hex centers
    :param layout: 'odd-r' for pointy-top hexagons
    :output: List of [lon, lat] coordinate pairs forming hexagon
    """
    
    # For pointy-topped hexagons (odd-r layout)
    # Calculate center position
    x_offset = width / 2 if r % 2 == 1 else 0
    center_x = q * width + x_offset
    center_y = r * height * 0.75

    
    # Generate 6 vertices of hexagon (pointy-top)
    # Starting from right vertex, going clockwise
    coords = []

    # Height of one "side" of the hexagon
    h = height / 2
    # Half-width
    w = width / 2
    
    # Generate 6 vertices in clockwise order, starting from top-right (negating 0.015 to stop the shifting of the map along x axis
    # between 2023-2024)
    vertices = [
        [center_x + w - 0.08, center_y + height / 4],      # Right-top (30°)
        [center_x - 0.08, center_y + h],                    # Top (90°)
        [center_x - w - 0.08, center_y + height / 4],      # Left-top (150°)
        [center_x - w - 0.08, center_y - height / 4],      # Left-bottom (210°)
        [center_x - 0.08, center_y - h],                    # Bottom (270°)
        [center_x + w - 0.08, center_y - height / 4]       # Right-bottom (330°)
    ]
    
    # Round to 5 decimal places to match example format
    coords = [[round(x, 5), round(y, 5)] for x, y in vertices]
    
    # Close the polygon by repeating first coordinate
    coords.append(coords[0])
    
    return coords

In [34]:
# convertin 2024 file from hexjson to geojson
hexjson_file_2024 = Path("../map_data/con_2024/uk-constituencies-2024.hexjson")

hexjson_to_geojson(
        hexjson_file_2024,
        '../map_data/con_2024/uk-constituencies-2024.geojson',
        hex_width=0.00234,
        hex_height=0.00270
    )

Converted 650 constituencies
Output saved to: ../map_data/con_2024/uk-constituencies-2024.geojson


In [35]:
# loading the geojson files
geo_2015_path = Path("../map_data/con_2015/constituency_results_2010_2019_hex.geojson")
geo_2024_path = Path("../map_data/con_2024/uk-constituencies-2024.geojson")

geo_2015 = gpd.read_file(geo_2015_path).rename(columns={"constituency_name":"Constituency"})
geo_2024 = gpd.read_file(geo_2024_path).rename(columns={"constituency_name":"Constituency"})

# fixing discrepancies in petitions constituency names and geojson constituency names
geo_2015 = geo_2015[['Constituency', 'geometry']]
geo_2015['Constituency'] = geo_2015['Constituency'].replace({
    "Stoke-On-Trent Central": "Stoke-on-Trent Central", 
    "Stoke-On-Trent South": "Stoke-on-Trent South",
    "Stoke-On-Trent North": "Stoke-on-Trent North",
    "Newcastle-Under-Lyme": "Newcastle-under-Lyme",
    "City Of Chester": "City of Chester",
    "Vale Of Clwyd": "Vale of Clwyd",
    "Ashton-Under-Lyne": "Ashton-under-Lyne",
    "City Of Durham": "City of Durham",
    "Newcastle Upon Tyne Central": "Newcastle upon Tyne Central",
    "Newcastle Upon Tyne North": "Newcastle upon Tyne North",
    "Newcastle Upon Tyne East": "Newcastle upon Tyne East",
    "Berwick-Upon-Tweed": "Berwick-upon-Tweed",
    "Na h-Eileanan An Iar": "Na h-Eileanan an Iar",
    "Forest Of Dean": "Forest of Dean",
    "Stratford-On-Avon": "Stratford-on-Avon",
    "Cities Of London and Westminster": "Cities of London and Westminster",
    "Isle Of Wight": "Isle of Wight",
    "Vale Of Glamorgan": "Vale of Glamorgan"
    })

geo_2024 = geo_2024[['Constituency', 'geometry']]

xmin, ymin, xmax, ymax = geo_2015["geometry"].total_bounds

# merging geojson with each year's petitions file
geo_file_2015 = geo_2015.set_index("Constituency").join(petitions_con_2015.set_index("Constituency"), how='outer').reset_index()
geo_file_2020 = geo_2015.set_index("Constituency").join(petitions_con_2020.set_index("Constituency"), how='outer').reset_index()
geo_file_2021 = geo_2015.set_index("Constituency").join(petitions_con_2021.set_index("Constituency"), how='outer').reset_index()
geo_file_2022 = geo_2015.set_index("Constituency").join(petitions_con_2022.set_index("Constituency"), how='outer').reset_index()
geo_file_2023 = geo_2015.set_index("Constituency").join(petitions_con_2023.set_index("Constituency"), how='outer').reset_index()
geo_file_2024 = geo_2024.set_index("Constituency").join(petitions_con_2024.set_index("Constituency"), how='outer').reset_index()
geo_file_2025 = geo_2024.set_index("Constituency").join(petitions_con_2025.set_index("Constituency"), how='outer').reset_index()

#creating a list of layers
layers = [geo_file_2021, geo_file_2022, geo_file_2023, geo_file_2024, geo_file_2025]


## Layers by year

In [36]:
# apply the function to each year creating 4 layers
layer1 = create_hex_layergroup_year(geo_file_2022, False, 8, "Asylum visas, Ukraine and Russia",2022)
layer2 = create_hex_layergroup_year(geo_file_2023, False, 16, "Healthcare and visas", 2023)
layer3 = create_hex_layergroup_year(geo_file_2024, False, 1, "Immigration, tax and general election", 2024)
layer4 = create_hex_layergroup_year(geo_file_2025, True, 6, "Immigration and visas", 2025)

# the layers control will automatically pick up on the layer groups
layer_control = fl.LayerControl(position="topleft")

# create map and add our components
map = fl.Map(tiles=None)

# had to recalibrate bounds a little
map.fit_bounds([[xmax-0.015, ymax+0.085], [xmin-0.175, ymin+0.085]])

layer1.add_to(map)
layer2.add_to(map)
layer3.add_to(map)
layer4.add_to(map)
layer_control.add_to(map)

# to display
map.save("hexmap_immigration.html")
webbrowser.open("hexmap_immigration.html")

True

## Layers by topic

In [37]:
# Now seperating layers by topic for a given year. Each layer is a different topic. 
layer1 = create_hex_layergroup_topic(geo_file_2020, False, 14, "Covid, furlough\n and financial support")
layer2 = create_hex_layergroup_topic(geo_file_2020, False, 13, 
                                     "Business (nurseries, schools)\nopening restrictions")
layer3 = create_hex_layergroup_topic(geo_file_2020, False, 6, 
                                     "Vaccination prioritsation and hair salon covid closure")
layer4 = create_hex_layergroup_topic(geo_file_2020, True, 12, 
                                     "Business (Fitness)\n reopening restrictions\nand face coverings")

# the layers control will automatically pick up on the layer groups
layer_control = fl.LayerControl(position="topleft")

# create map and add our components
map = fl.Map(tiles=None)
map.fit_bounds([[xmax-0.015, ymax+0.085], [xmin-0.175, ymin+0.085]])

layer1.add_to(map)
layer2.add_to(map)
layer3.add_to(map)
layer4.add_to(map)
layer_control.add_to(map)

# to display
map.save("hexmap_covid_2020.html")
webbrowser.open("hexmap_covid_2020.html")

True

In [38]:
# Again covid based topics but for 2021 rather than 2020
layer1 = create_hex_layergroup_topic(geo_file_2021, False, 10, "Covid travel and masks")
layer2 = create_hex_layergroup_topic(geo_file_2021, False, 15, "Vaccination programme")
layer3 = create_hex_layergroup_topic(geo_file_2021, False, 9, "University tuition\nand accomodation refunds")
layer4 = create_hex_layergroup_topic(geo_file_2021, True, 2, "Cancelling exams")

# the layers control will automatically pick up on the layer groups
layer_control = fl.LayerControl(position="topleft")

# create map and add our components
map = fl.Map(tiles=None)
map.fit_bounds([[xmax-0.015, ymax+0.085], [xmin-0.175, ymin+0.085]])

layer1.add_to(map)
layer2.add_to(map)
layer3.add_to(map)
layer4.add_to(map)
layer_control.add_to(map)

# to display
map.save("hexmap_covid_2021.html")
webbrowser.open("hexmap_covid_2021.html")

True